# 05 — Convolutional Neural Networks — Hands-on Tutorial

In this notebook you will:
- Step a 1D filter across a synthetic stellar spectrum and watch the output accumulate
- Apply 2D convolution to a synthetic galaxy image with edge, blur, and sharpening kernels
- Visualise what each filter detects — the feature map view of a convolutional layer
- Work through the parameter count arithmetic from the slide deck
- Trace spatial dimensions through max and average pooling

**Prior:** Notebooks 01–04 complete; slides 04 (convolutional neural networks)

## Where this fits

| # | Topic | Slide deck | Notebook |
|---|---|---|---|
| 1 | Single neuron | `01_single_neuron.pdf` | `01_single_neuron.ipynb` |
| 2 | Multilayer networks | `02_multilayer_networks.pdf` | `02_training_loop.ipynb` |
| 3 | Backpropagation | `03_backprop_training.pdf` | `03_backpropagation.ipynb` |
| 4 | Optimizers | `07_optimizers.pdf` *(new)* | `04_optimizers.ipynb` |
| **→ 5** | **CNNs** | **`04_cnns.pdf`** | **`05_cnns.ipynb`** |
| 6 | Modern architectures | `05a_attention.pdf` + `05b_practical.pdf` | *(bonus: `bonus_generative_models.ipynb`)* |
| 7 | Bayesian inference | `06_bayesian_inference.pdf` | `06_bayesian_inference.ipynb` |

**Coming from:** All previous notebooks worked with vector inputs; this one moves to images and the convolution operator.

**Leading to:** The modern-architectures deck builds on the spatial intuitions (filters, receptive fields, pooling) you develop here.

**If you skipped ahead:** You need to be comfortable with `nn.Linear` and the basic training loop from notebooks 02 through 04.


In [ ]:
# ── Environment setup ──────────────────────────────────────────────────────
# Run this cell only on Colab. On JupyterHub, packages are pre-installed.
import sys
if 'google.colab' in sys.modules:
    %pip install -q ipywidgets torch

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
from ipywidgets import interact, IntSlider, Dropdown

%matplotlib inline

---

## The mathematics of convolution

### 1D convolution

$$y_i = \sum_{j=0}^{k-1} s_{i+j} \cdot f_j, \qquad i = 0, 1, \ldots, L_{\text{out}}-1$$

### 2D convolution

$$y_{i,j} = \sum_{p=0}^{k-1} \sum_{q=0}^{k-1} X_{i+p,\, j+q} \cdot K_{p,q}$$

| Symbol | Meaning |
|--------|---------|
| $s$ | Input signal (e.g. stellar spectrum of length $L$) |
| $f$ | Filter / kernel of length $k$ |
| $y_i$ | Output at position $i$ — the filter's response at that location |
| $X$ | Input image (2D array) |
| $K$ | 2D kernel of size $k \times k$ |
| $y_{i,j}$ | Output feature map value at position $(i,j)$ |

### Output size and parameter count

$$L_{\text{out}} = L_{\text{in}} - k + 1 \quad \text{(no padding)}, \qquad L_{\text{out}} = L_{\text{in}} \quad \text{(same padding)}$$

$$\text{Parameters in Conv2d}(C_{\text{in}}, C_{\text{out}}, k) = C_{\text{out}} \times C_{\text{in}} \times k^2 + C_{\text{out}}$$

The parameter count does **not** depend on the input image size. This is parameter sharing — the same kernel slides over the entire image.

In [ ]:
# ── Implement 1D convolution as an explicit loop; plot one kernel pass ────────

def conv1d_loop(signal, kernel):
    """
    Slide kernel over signal, computing the dot product at each position.
    Returns output of length len(signal) - len(kernel) + 1.
    """
    k   = len(kernel)
    out = []
    for i in range(len(signal) - k + 1):
        patch = signal[i : i + k]           # slice of length k
        value = np.sum(patch * kernel)       # dot product: the convolution sum
        out.append(value)
    return np.array(out)

# Build a simple synthetic spectrum
np.random.seed(7)
x_sig  = np.linspace(0, 1, 40)
signal = (1 - 0.15*x_sig
          - 0.55 * np.exp(-((x_sig - 0.45)**2) / (2*0.018**2))
          + 0.04 * np.random.randn(40))

kernels = {
    'edge  $[1, 0, -1]$':         np.array([ 1,  0, -1], float),
    'smooth  $[1/3, 1/3, 1/3]$':  np.array([ 1,  1,  1], float) / 3,
    'line detect  $[-1, 2, -1]$': np.array([-1,  2, -1], float),
}

fig, axes = plt.subplots(len(kernels), 2, figsize=(12, 9))

for row, (name, k) in enumerate(kernels.items()):
    output = conv1d_loop(signal, k)

    # Left: signal with kernel position highlighted at peak response
    peak = int(np.argmax(np.abs(output)))
    axes[row, 0].plot(signal, 'b-o', ms=3, lw=1.5, label='Signal')
    axes[row, 0].fill_between(range(peak, peak+len(k)),
                               signal[peak:peak+len(k)], alpha=0.4, color='orange',
                               label=f'Kernel window at peak (pos {peak})')
    axes[row, 0].set_title(f'Kernel: {name}')
    axes[row, 0].legend(fontsize=8); axes[row, 0].set_ylabel('Flux')

    # Right: output
    axes[row, 1].plot(output, 'r-o', ms=3, lw=1.5)
    axes[row, 1].axhline(0, color='gray', lw=0.7, ls='--')
    axes[row, 1].scatter([peak], [output[peak]], c='orange', s=80, zorder=5,
                          label=f'Peak response = {output[peak]:.3f}')
    axes[row, 1].set_title(f'Convolution output  (length = {len(output)})')
    axes[row, 1].legend(fontsize=8); axes[row, 1].set_ylabel('Response')

for ax in axes[-1]: ax.set_xlabel('Position')

plt.suptitle('1D convolution: the same sum formula, three different kernels', y=1.01)
plt.tight_layout(); plt.show()

# Output size formula
print('Output size formula:  L_out = L_in − k + 1')
print(f'  L_in={len(signal)}, k=3  →  L_out = {len(signal)-3+1}')
print()
print('Parameter count formula:  C_out × C_in × k² + C_out')
print(f'  Conv2d(1→8, k=3):  8 × 1 × 9 + 8 = {8*1*9+8} parameters')
print(f'  Conv2d(1→8, k=5):  8 × 1 × 25 + 8 = {8*1*25+8} parameters')
print(f'  Conv2d(8→16, k=3): 16 × 8 × 9 + 16 = {16*8*9+16} parameters')


### Think about it

- The `edge` kernel $[1, 0, -1]$ computes the *difference* between the signal   value two positions apart. Where in the stellar spectrum does it respond most   strongly — at the absorption line edges, the continuum, or the noise peaks?
- The output length is $L_{\text{in}} - k + 1$ without padding. If you stack two   $k=3$ layers on a signal of length 40, what is the output length after both layers?   After three layers?
- A `Conv2d(1, 32, kernel_size=3)` layer has $32 \times 1 \times 9 + 32 = 320$ parameters,   regardless of input image size. A fully-connected layer connecting a $64\times64$   image to 32 neurons has how many parameters? What is the ratio?
- The convolution sum $\sum_j s_{i+j} \cdot f_j$ is mathematically identical to a   dot product between the signal patch and the kernel. What does a large positive   dot product tell you about the similarity between the patch and the kernel?

---

## Part 1 — Convolution by hand: 1D spectrum

A convolution filter slides across a signal, computing a weighted sum at each position.
For a signal $s$ of length $L$ and a filter $k$ of length $n$, the output at position $i$ is:

$$y_i = \sum_{j=0}^{n-1} s_{i+j} \cdot k_j$$

The output length is $L_{out} = L_{in} - n + 1$ (with no padding).

Below: a synthetic stellar spectrum with a Gaussian absorption line and continuum noise.
Drag the **position** slider to step the kernel across the spectrum one step at a time,
and watch the output build up in the lower panel.

**Kernels to try:**
- `edge`: detects sharp transitions — derivative of the spectrum
- `smooth`: averages neighbouring bins — suppresses noise
- `line_detect`: responds to narrow absorption or emission features

In [2]:
# ── Part 1: 1D convolution step-through ─────────────────────────────────────

def make_spectrum(n=40):
    """Synthetic stellar spectrum: sloped continuum + Gaussian absorption line + noise."""
    x = np.linspace(0, 1, n)
    continuum = 1.0 - 0.15 * x
    line = -0.55 * np.exp(-((x - 0.45)**2) / (2 * 0.018**2))
    noise = 0.04 * np.random.RandomState(7).randn(n)
    return continuum + line + noise

KERNELS_1D = {
    'edge':        np.array([ 1,  0, -1], dtype=float),
    'smooth':      np.array([ 1,  1,  1], dtype=float) / 3,
    'line_detect': np.array([-1,  2, -1], dtype=float),
}

SPECTRUM = make_spectrum(40)

def plot_conv1d_step(kernel_choice='line_detect', position=0):
    k = KERNELS_1D[kernel_choice]
    k_len = len(k)
    n = len(SPECTRUM)
    out_len = n - k_len + 1
    pos = int(np.clip(position, 0, out_len - 1))

    full_output = np.array(
        [np.dot(SPECTRUM[i:i + k_len], k) for i in range(out_len)]
    )
    current_val = full_output[pos]

    fig, axes = plt.subplots(2, 1, figsize=(10, 6))

    # Top panel: spectrum + kernel window
    axes[0].plot(SPECTRUM, 'b-o', ms=4, lw=1.5, label='Spectrum')
    patch_x = np.arange(pos, pos + k_len)
    axes[0].fill_between(patch_x, SPECTRUM[patch_x], alpha=0.3, color='orange')
    axes[0].plot(patch_x, SPECTRUM[patch_x], 'o', c='orange', ms=9, label='Kernel window')
    axes[0].set_title(
        f'Kernel: {kernel_choice}  {list(k)}  |  window at position {pos}  '
        f'|  dot product = {current_val:.3f}'
    )
    axes[0].set_xlabel('Wavelength bin')
    axes[0].set_ylabel('Flux')
    axes[0].legend()

    # Bottom panel: output so far
    axes[1].plot(full_output, 'r--', alpha=0.25, lw=1.5, label='Full output (preview)')
    axes[1].plot(np.arange(pos + 1), full_output[:pos + 1], 'r-o', ms=5, lw=2,
                 label='Output built so far')
    axes[1].scatter([pos], [current_val], c='orange', s=120, zorder=5,
                    label=f'Current: {current_val:.3f}')
    axes[1].axhline(0, c='gray', lw=0.7, ls='--')
    axes[1].set_title(f'Convolution output  (output length = {out_len})')
    axes[1].set_xlabel('Output position')
    axes[1].set_ylabel('Response')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

interact(
    plot_conv1d_step,
    kernel_choice=Dropdown(options=['edge', 'smooth', 'line_detect'], description='Kernel'),
    position=IntSlider(value=0, min=0, max=36, step=1, description='Position'),
);

interactive(children=(Dropdown(description='Kernel', options=('edge', 'smooth', 'line_detect'), value='edge'),…

### Think about it

- At what position does `line_detect` give its largest positive response? Does this
  correspond to the centre of the absorption line, or to one of its edges?
- What does the `smooth` kernel output represent physically? If you applied it many times
  in sequence, what would the spectrum eventually look like?
- The `edge` kernel has values `[1, 0, -1]`. At a position where the spectrum is perfectly
  flat, what is the output? What about at a position straddling the start of the absorption dip?
- The output length is $L_{out} = L_{in} - 3 + 1 = 38$ for a length-40 signal and a 3-element
  kernel. How would you pad the signal so that $L_{out} = L_{in} = 40$? What value would you pad with?

---

## Part 2 — 2D convolution on a galaxy image

The same principle extends to 2D: a $k \times k$ kernel slides over an image and
computes the inner product with each $k \times k$ patch at every position.

The galaxy below is generated synthetically: a Gaussian bulge, an exponential disc,
and a spiral pattern. Each kernel reveals a different aspect of the image structure.

*If galaxies aren't your thing: think 'small image, structured local patterns' — the same CNN reasoning applies to MNIST digits or medical image patches.*

**What each kernel does:**
- `edge_h` / `edge_v`: detect horizontal or vertical brightness transitions
- `blur`: averages a $5 \times 5$ neighbourhood — removes fine detail
- `sharpen`: amplifies local contrast
- `centre_surround`: fires where a bright centre is surrounded by darker pixels —
  analogous to on-centre retinal cells, and similar to first-layer CNN filters
  trained on natural images

In [ ]:
# ── Part 2: 2D convolution on a synthetic galaxy ─────────────────────────────

def make_galaxy(size=64):
    """Synthetic galaxy: Gaussian bulge + exponential disc + spiral modulation."""
    x = np.linspace(-3, 3, size)
    y = np.linspace(-3, 3, size)
    X, Y = np.meshgrid(x, y)
    r = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)
    bulge  = np.exp(-r**2 / 0.4)
    disc   = 0.5 * np.exp(-r / 2.0)
    spiral = 0.35 * disc * np.maximum(0, np.cos(2 * theta - r * 1.5))
    rng = np.random.RandomState(42)
    img = bulge + disc + spiral + 0.04 * rng.randn(size, size)
    img = np.clip(img, 0, None)
    img /= img.max()
    return img

GALAXY = make_galaxy(64)

KERNELS_2D = {
    'edge_h':         np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=float) / 3,
    'edge_v':         np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=float) / 3,
    'blur':           np.ones((5, 5), dtype=float) / 25,
    'sharpen':        np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=float),
    'centre_surround': np.array([[0,-1,0],[-1,4,-1],[0,-1,0]], dtype=float) / 4,
}

def plot_conv2d(kernel_choice='edge_h'):
    k = KERNELS_2D[kernel_choice]
    img_t = torch.tensor(GALAXY, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    k_t   = torch.tensor(k,      dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    pad   = k.shape[0] // 2
    out   = F.conv2d(img_t, k_t, padding=pad).squeeze().numpy()

    cmap_out = 'RdBu_r' if 'edge' in kernel_choice or 'centre' in kernel_choice else 'inferno'

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(GALAXY, cmap='inferno')
    axes[0].set_title('Input: synthetic galaxy  (64x64)')
    axes[0].axis('off')

    axes[1].imshow(out, cmap=cmap_out)
    axes[1].set_title(f'After {kernel_choice} kernel  (shape {k.shape})')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()
    print(f'Input  range: [{GALAXY.min():.3f}, {GALAXY.max():.3f}]')
    print(f'Output range: [{out.min():.3f}, {out.max():.3f}]')

interact(
    plot_conv2d,
    kernel_choice=Dropdown(
        options=['edge_h', 'edge_v', 'blur', 'sharpen', 'centre_surround'],
        description='Kernel'),
);

### Think about it

- Apply `edge_h` and then `edge_v` mentally. Where do both fire strongly? What does
  the *combination* of horizontal and vertical edges detect?
- The `blur` output range is smaller than the input range. Why? What information
  has been permanently removed from the image?
- The `centre_surround` kernel has a negative surround. The beam response of some
  radio telescope deconvolution methods also has a negative outer ring (negative bowl).
  Can you see why the same filter concept applies?
- For the `sharpen` kernel, the output can exceed 1.0. Why does this happen? What
  does this mean for training a deep network without normalisation between layers?

---

## Part 3 — Feature maps: seeing what each filter detects

A convolutional layer applies multiple filters to the input *simultaneously*.
Each filter produces one **feature map** — a 2D array encoding where that filter
fires strongly across the image.

The plot below applies five filters in parallel to the synthetic galaxy.
This is exactly what the first layer of a trained CNN does,
except the filters are *learned from data* rather than hand-designed.

Zeiler & Fergus (2013) showed that for networks trained on natural images,
the first-layer learned filters look strikingly similar to these:
edges at different orientations, blobs, and centre–surround patterns.

In [ ]:
# ── Part 3: Feature maps — all five filters applied in parallel ──────────────

img_t = torch.tensor(GALAXY, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))

# Input
axes[0, 0].imshow(GALAXY, cmap='inferno')
axes[0, 0].set_title('Input (original)', fontsize=11)
axes[0, 0].axis('off')

cmaps = {
    'edge_h': 'RdBu_r', 'edge_v': 'RdBu_r',
    'blur': 'inferno', 'sharpen': 'inferno', 'centre_surround': 'RdBu_r',
}

for ax, (name, k) in zip(axes.flat[1:], KERNELS_2D.items()):
    k_t = torch.tensor(k, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    pad = k.shape[0] // 2
    out = F.conv2d(img_t, k_t, padding=pad).squeeze().numpy()
    ax.imshow(out, cmap=cmaps[name])
    ax.set_title(name, fontsize=10)
    ax.axis('off')

axes.flat[-1].axis('off')   # sixth panel unused

plt.suptitle(
    'Feature maps: one filter = one detection channel\n'
    'A trained CNN first layer learns these automatically',
    fontsize=12
)
plt.tight_layout()
plt.show()

### Think about it

- The `edge_h` and `edge_v` maps look like rotated versions of each other.
  Why might a trained CNN still learn both, rather than learning a single diagonal-edge filter?
- The `blur` feature map is the most similar to the original galaxy.
  At what layer depth would you expect blur-like filters to stop being useful for
  classifying galaxy morphology?
- A filter with all-positive weights acts like a **local average** (photometer).
  A `centre_surround` filter acts like a **local contrast detector**.
  Which is better for detecting the boundary between a galaxy bulge and disc?
- After Layer 1 computes these feature maps, Layer 2 receives them as inputs — not pixels.
  What types of *combinations* of edges and centre–surround responses might Layer 2
  learn to detect for classifying spirals vs ellipticals?

---

## Part 4 — Parameter count: fully-connected vs convolutional

The slide deck opened with the parameter count argument.
Here is the arithmetic in runnable code.

In [ ]:
# ── Part 4: Parameter count arithmetic ──────────────────────────────────────

print('=== Fully-connected network, first layer (Galaxy10: 256x256x3 input) ===')
fc_in  = 256 * 256 * 3    # 196,608 inputs
fc_out = 512
fc_w   = fc_in * fc_out
fc_b   = fc_out
fc_total = fc_w + fc_b
print(f'  Input:      {fc_in:>12,}  ({256}x{256}x{3})')
print(f'  Output:     {fc_out:>12,}  neurons')
print(f'  Weights:    {fc_w:>12,}')
print(f'  Biases:     {fc_b:>12,}')
print(f'  TOTAL:      {fc_total:>12,}')

print()
print('=== Convolutional network, first layer (3x3 kernel, 3 ch in, 32 filters) ===')
c_in, c_out, k = 3, 32, 3
conv_w   = c_in * c_out * k * k
conv_b   = c_out
conv_total = conv_w + conv_b
print(f'  Filter:     {k}x{k}, {c_in} channels in, {c_out} filters')
print(f'  Weights:    {k}x{k}x{c_in}x{c_out} = {conv_w:>8,}')
print(f'  Biases:     {conv_b:>12,}')
print(f'  TOTAL:      {conv_total:>12,}')
print(f'  (These {conv_total} weights scan every position of a {256}x{256} image.)')

print()
ratio = fc_total / conv_total
print(f'Ratio: the FC first layer has {ratio:,.0f}x more parameters than the Conv first layer.')

print()
print('=== Full GalaxyCNN: three conv blocks ===')
layers = [
    ('Conv2d(3,  32, 3)', 3 * 32 * 3 * 3 + 32),
    ('Conv2d(32, 64, 3)', 32 * 64 * 3 * 3 + 64),
    ('Conv2d(64,128, 3)', 64 * 128 * 3 * 3 + 128),
    ('Linear(128*32*32, 256)', 128 * 32 * 32 * 256 + 256),
    ('Linear(256, 10)', 256 * 10 + 10),
]
total_cnn = 0
for name, n in layers:
    print(f'  {name:<30}  {n:>10,} params')
    total_cnn += n
print(f'  {"TOTAL":<30}  {total_cnn:>10,} params')
print(f'  (vs {fc_total:,} params for the first FC layer alone)')

### Think about it

- The parameter count for `Conv2d` does **not** depend on the input image size — only
  on kernel size, input channels, and number of filters. Why is this a significant
  advantage when applying a model trained on 64x64 images to 256x256 images?
- In the GalaxyCNN above, which layer dominates the parameter count? Is this
  the conv layers or the classifier head? What does this tell you about where most
  of the network's capacity lives?
- A 1x1 convolution (`kernel_size=1`) has how many parameters for 64 input channels
  and 128 output channels? What does a 1x1 conv compute, and why is it useful?

---

## Part 5 — Pooling and spatial reduction

Pooling reduces spatial resolution deliberately.
It replaces a $k \times k$ block of values with a single summary statistic.

- **MaxPool:** takes the maximum — keeps the strongest activation in each region
- **AvgPool:** takes the mean — smooths the feature map

After pooling, the network no longer knows *exactly* where a feature was within a region,
only *whether* it was there. This is a deliberate trade-off:
small position shifts should not change the galaxy class.

Spatial dimension after each MaxPool2d(2,2):

$$256 \;\rightarrow\; 128 \;\rightarrow\; 64 \;\rightarrow\; 32$$

Use the sliders to see this on the synthetic galaxy.

In [ ]:
# ── Part 5: Pooling and spatial reduction ────────────────────────────────────

def plot_pooling(pool_type='max', pool_size=2, n_pools=1):
    ks = int(pool_size)
    n  = int(n_pools)
    img_t   = torch.tensor(GALAXY, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    current = img_t
    sizes   = [64]

    for _ in range(n):
        if pool_type == 'max':
            current = F.max_pool2d(current, kernel_size=ks, stride=ks)
        else:
            current = F.avg_pool2d(current, kernel_size=ks, stride=ks)
        sizes.append(sizes[-1] // ks)

    pooled = current.squeeze().numpy()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
    ax1.imshow(GALAXY, cmap='inferno')
    ax1.set_title('Input: 64x64')
    ax1.axis('off')

    ax2.imshow(pooled, cmap='inferno')
    ax2.set_title(
        f'After {n}x {pool_type.capitalize()}Pool2d({ks},{ks}): {sizes[-1]}x{sizes[-1]}'
    )
    ax2.axis('off')

    n_in  = 64 * 64
    n_out = sizes[-1] * sizes[-1]
    plt.suptitle(
        f'Spatial reduction: {64}x{64} ({n_in:,} locations) '
        f'-> {sizes[-1]}x{sizes[-1]} ({n_out:,} locations)',
        fontsize=10
    )
    plt.tight_layout()
    plt.show()
    print('Dimension trace: ' + ' -> '.join(str(s) for s in sizes))

interact(
    plot_pooling,
    pool_type=Dropdown(options=[('MaxPool', 'max'), ('AvgPool', 'avg')], description='Pool type'),
    pool_size=Dropdown(options=[2, 4], description='Pool size'),
    n_pools=Dropdown(options=[1, 2, 3], description='Rounds'),
);

### Think about it

- After 3 rounds of MaxPool2d(2,2) on a 256x256 image, what is the spatial resolution?
  If the galaxy image covers 50 arcseconds at 0.25 arcsec/pixel, how many arcseconds
  does each spatial location in the final feature map correspond to?
- MaxPool always selects the highest-value pixel in a region. For a faint galaxy on a
  noisy bright background, could this cause problems? What would AvgPool do differently?
- Set pool_size=4, n_pools=3. The output is 1x1 — a single value per feature map.
  What is the classifier head seeing at that point? Is the spatial structure of the
  galaxy still recoverable?

---

## Part 6 — Receptive field: how deep networks see further

A single convolutional layer with a $3 \times 3$ kernel looks at a $3 \times 3$ patch of the input. Stack two such layers: each neuron in layer 2 depends on a $3 \times 3$ region of layer 1, which in turn each depends on a $3 \times 3$ region of the input — so a layer-2 neuron sees a **$5 \times 5$** patch of the input.

This growth is the mechanism that lets deep CNNs detect large-scale structure (galaxy morphology, extended emission) while using only small local kernels. Pooling accelerates this growth even further.

**Receptive field for $n$ stacked $3\times3$ layers (no pooling):**
$$RF(n) = 2n + 1$$

**With a MaxPool(2,2) between every two conv layers:**
$$RF \approx 2^{\lfloor n/2 \rfloor} \times (\text{base growth})$$

In [ ]:
# ── Part 6: receptive field growth with depth ────────────────────────────────

def compute_receptive_field(config):
    """
    Compute receptive field size given a list of layer configs.
    Each entry is ('conv', kernel_size) or ('pool', stride).
    Returns (rf_size, effective_stride) after all layers.
    """
    rf, stride = 1, 1
    for layer_type, k in config:
        if layer_type == 'conv':
            rf     = rf + (k - 1) * stride
            # stride unchanged for conv with stride=1
        elif layer_type == 'pool':
            stride = stride * k   # pooling multiplies effective stride
    return rf

# ── Three architecture profiles ──────────────────────────────────────────────
max_depth = 10

# Profile A: stacked 3x3 convolutions only
rf_conv_only  = [compute_receptive_field([('conv', 3)] * n) for n in range(1, max_depth+1)]

# Profile B: 3x3 conv + MaxPool(2) every 2 conv layers
def profile_b(n_conv):
    config = []
    for i in range(1, n_conv+1):
        config.append(('conv', 3))
        if i % 2 == 0:
            config.append(('pool', 2))
    return compute_receptive_field(config)
rf_with_pool  = [profile_b(n) for n in range(1, max_depth+1)]

# Profile C: 5x5 convolutions only
rf_5x5        = [compute_receptive_field([('conv', 5)] * n) for n in range(1, max_depth+1)]

depths = list(range(1, max_depth+1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(depths, rf_conv_only,  'o-', label='3×3 conv only',          lw=2)
ax1.plot(depths, rf_with_pool,  's-', label='3×3 conv + MaxPool(2)/2', lw=2)
ax1.plot(depths, rf_5x5,        '^-', label='5×5 conv only',           lw=2)
ax1.axhline(64,  color='gray', ls='--', lw=0.8, label='Galaxy image size (64px)')
ax1.axhline(256, color='red',  ls='--', lw=0.8, label='Galaxy10 size (256px)')
ax1.set_xlabel('Number of convolutional layers')
ax1.set_ylabel('Receptive field size (pixels)')
ax1.set_title('Receptive field growth with depth')
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

# Visualise what different receptive fields "see" on the galaxy image
rf_sizes = [3, 7, 15, 31]
n_show   = len(rf_sizes)
ax2.imshow(GALAXY, cmap='inferno', extent=[-32, 32, -32, 32])
center   = 0
colours  = plt.cm.cool(np.linspace(0.2, 0.9, n_show))
for rf, col in zip(rf_sizes, colours):
    half = rf / 2
    rect = plt.Rectangle((-half, -half), rf, rf,
                         linewidth=2, edgecolor=col, facecolor='none',
                         label=f'RF={rf}px  ({rf}×{rf})')
    ax2.add_patch(rect)
ax2.set_title('Receptive field sizes on the galaxy\n(centred at image centre)')
ax2.legend(fontsize=8, loc='upper right'); ax2.axis('off')

plt.tight_layout()
plt.show()

print('Layers needed to cover the 64×64 galaxy image:')
for label, rfs in [('3×3 only', rf_conv_only), ('3×3 + pool', rf_with_pool)]:
    n = next((i+1 for i, r in enumerate(rfs) if r >= 64), None)
    print(f'  {label:20s}: {n} layers')


### Think about it

- How many stacked $3\times3$ conv layers are needed to cover the entire $64\times64$   galaxy image? How does pooling change that number?
- A $7\times7$ kernel has the same receptive field as **three** stacked $3\times3$   layers but with more parameters. Why do modern architectures prefer stacked   small kernels?
- VGG-16 (a classic architecture) uses 13 conv layers of $3\times3$.   What is its theoretical receptive field? Does that match the ImageNet image size   of 224px?
- The spiral arms of the synthetic galaxy are large-scale structures (~20–30px).   How many layers would a CNN need before any single neuron "sees" a full spiral arm?

---

## Part 7 — MLP vs CNN on real galaxies (the payoff)

Everything so far was mechanics. Now the point: on the **same** Galaxy10 images we
train a fully-connected **MLP** and a **CNN**, and see two things the lecture
promised —

1. the CNN reaches the same (or better) accuracy with **orders of magnitude fewer
   parameters**, and
2. when we **move the galaxy around the frame**, the MLP falls apart while the CNN
   barely notices — *translational invariance*, measured.

> **Speed:** set `FAST = True` (16x16 galaxy on a 32x32 canvas) for a quick run.
> `FAST = False` uses 32x32 on a 64x64 canvas — prettier, roughly 4x slower.

In [ ]:
# ── Part 7: load Galaxy10, build a "galaxy on a canvas" dataset ──────────────
import os, urllib.request, h5py
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

# ---- speed knob ------------------------------------------------------------
# FAST: 16x16 galaxy on a 32x32 canvas  (trains in ~1-2 min on Colab CPU)
# SLOW: 32x32 galaxy on a 64x64 canvas  (prettier, ~4x slower)
FAST = True
GAL    = 16 if FAST else 32        # galaxy size (pixels)
CANVAS = 2 * GAL                   # model input size (galaxy can sit anywhere)
N_SUB  = 3000                      # subset of Galaxy10 for a quick demo
EPOCHS = 8
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"GAL={GAL}  CANVAS={CANVAS}  subset={N_SUB}  device={DEVICE}")

# ---- download Galaxy10 (SDSS, 69x69x3, ~200 MB) ----------------------------
URL  = "https://astro.utoronto.ca/~hleung/shared/Galaxy10/Galaxy10.h5"
PATH = "Galaxy10.h5"
if not os.path.exists(PATH):
    print("downloading Galaxy10 (~200 MB, one-time)...")
    urllib.request.urlretrieve(URL, PATH)
with h5py.File(PATH, "r") as f:
    imgs_raw = np.asarray(f["images"], dtype=np.float32) / 255.0   # (N,69,69,3)
    lbls_raw = np.asarray(f["ans"], dtype=np.int64)                # (N,)
print("Galaxy10:", imgs_raw.shape, "  classes:", np.unique(lbls_raw))

# ---- take a class-balanced subset, resize each galaxy to GAL x GAL ---------
rng = np.random.RandomState(0)
sel = rng.choice(len(imgs_raw), size=N_SUB, replace=False)
imgs = torch.from_numpy(imgs_raw[sel]).permute(0, 3, 1, 2)          # (n,3,69,69)
lbls = torch.from_numpy(lbls_raw[sel])
gal  = F.interpolate(imgs, size=(GAL, GAL), mode='bilinear', align_corners=False)

def paste_on_canvas(gal_batch, offsets):
    """Place each GALxGAL galaxy onto a CANVASxCANVAS black frame at (dy,dx)."""
    n = gal_batch.shape[0]
    canvas = torch.zeros(n, 3, CANVAS, CANVAS)
    for i, (dy, dx) in enumerate(offsets):
        canvas[i, :, dy:dy+GAL, dx:dx+GAL] = gal_batch[i]
    return canvas

# Training/eval "centred" set: every galaxy in the middle of the canvas.
c = (CANVAS - GAL) // 2
X_centred = paste_on_canvas(gal, [(c, c)] * len(gal))

# train / test split
n_tr = int(0.8 * len(gal))
Xtr, ytr = X_centred[:n_tr], lbls[:n_tr]
Xte, yte = X_centred[n_tr:], lbls[n_tr:]
gal_te   = gal[n_tr:]                      # keep raw test galaxies for shifting later

train_dl = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)
test_dl  = DataLoader(TensorDataset(Xte, yte), batch_size=128)
print(f"train {len(Xtr)}  test {len(Xte)}  input shape {tuple(Xtr.shape[1:])}")

### Two models, one task — count the parameters first

Before training, just build both and print their sizes. This is the Part 4
arithmetic, now on a real model.

In [ ]:
# ── Two models for the SAME task; print the parameter counts side by side ────
N_CLASSES = 10

class GalaxyMLP(nn.Module):
    """Fully-connected: every pixel wired to every hidden unit."""
    def __init__(self, canvas=CANVAS, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * canvas * canvas, hidden), nn.ReLU(),
            nn.Linear(hidden, N_CLASSES),
        )
    def forward(self, x): return self.net(x)

class GalaxyCNN(nn.Module):
    """Convolutional: the same small filters scan every position."""
    def __init__(self, canvas=CANVAS):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # C -> C/2
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # C/2 -> C/4
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * (canvas // 4) * (canvas // 4), N_CLASSES),
        )
    def forward(self, x): return self.head(self.features(x))

def count_params(m): return sum(p.numel() for p in m.parameters())

mlp, cnn = GalaxyMLP(), GalaxyCNN()
p_mlp, p_cnn = count_params(mlp), count_params(cnn)
print(f"MLP parameters: {p_mlp:>12,}")
print(f"CNN parameters: {p_cnn:>12,}")
print(f"\nThe MLP has {p_mlp / p_cnn:.0f}x more parameters than the CNN —")
print(f"and almost all of them live in the very first Linear layer:")
first_linear = 3 * CANVAS * CANVAS * 256
print(f"  first MLP layer alone: {first_linear:,}  "
      f"({100 * first_linear / p_mlp:.0f}% of the MLP)")

### Train both on centred galaxies

A few epochs of Adam. The CNN should match or beat the MLP despite being tiny.

In [ ]:
# ── Train both models on the centred galaxies; plot validation accuracy ──────
def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        pred = model(X.to(DEVICE)).argmax(1).cpu()
    return (pred == y).float().mean().item()

def train(model, epochs=EPOCHS):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    lossf = nn.CrossEntropyLoss()
    hist = []
    for ep in range(epochs):
        model.train()
        for xb, yb in train_dl:
            opt.zero_grad()
            loss = lossf(model(xb.to(DEVICE)), yb.to(DEVICE))
            loss.backward(); opt.step()
        acc = evaluate(model, Xte, yte)
        hist.append(acc)
        print(f"  epoch {ep+1:>2}/{epochs}   val acc = {acc:.3f}")
    return hist

torch.manual_seed(0)
mlp, cnn = GalaxyMLP(), GalaxyCNN()
print("Training MLP..."); h_mlp = train(mlp)
print("Training CNN..."); h_cnn = train(cnn)

plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS+1), h_mlp, 'o-', label=f'MLP  ({p_mlp:,} params)')
plt.plot(range(1, EPOCHS+1), h_cnn, 's-', label=f'CNN  ({p_cnn:,} params)')
plt.xlabel('epoch'); plt.ylabel('validation accuracy'); plt.ylim(0, 1)
plt.title('Same galaxies, centred: CNN matches/beats MLP with far fewer params')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"\nFinal (centred) accuracy — MLP: {h_mlp[-1]:.3f}   CNN: {h_cnn[-1]:.3f}")

### Now move the galaxy — the translational-invariance test

The models only ever saw **centred** galaxies. Put the same test galaxies at
**random positions** and re-measure. This is the whole reason CNNs exist for imaging.

In [ ]:
# ── The translational-invariance test: move the galaxy, re-measure ──────────
# Both models were trained ONLY on centred galaxies. Now put the SAME test
# galaxies at random positions on the canvas and re-evaluate.
rng = np.random.RandomState(1)
max_off = CANVAS - GAL
offsets = [(rng.randint(0, max_off + 1), rng.randint(0, max_off + 1))
           for _ in range(len(gal_te))]
X_shifted = paste_on_canvas(gal_te, offsets)

acc_mlp_c, acc_cnn_c = evaluate(mlp, Xte, yte),       evaluate(cnn, Xte, yte)
acc_mlp_s, acc_cnn_s = evaluate(mlp, X_shifted, yte), evaluate(cnn, X_shifted, yte)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].imshow(X_shifted[0].permute(1, 2, 0).numpy())
ax[0].set_title('same galaxy, now off-centre'); ax[0].axis('off')

x = np.arange(2); w = 0.35
ax[1].bar(x - w/2, [acc_mlp_c, acc_mlp_s], w, label='MLP', color='C3')
ax[1].bar(x + w/2, [acc_cnn_c, acc_cnn_s], w, label='CNN', color='C0')
ax[1].set_xticks(x); ax[1].set_xticklabels(['centred', 'shifted'])
ax[1].set_ylabel('test accuracy'); ax[1].set_ylim(0, 1)
ax[1].set_title('Move the galaxy: the MLP collapses, the CNN holds')
ax[1].legend(); ax[1].grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

print(f"           centred   shifted")
print(f"  MLP      {acc_mlp_c:.3f}     {acc_mlp_s:.3f}   <- drops: it memorised pixel positions")
print(f"  CNN      {acc_cnn_c:.3f}     {acc_cnn_s:.3f}   <- holds: same filters scan every position")

### Think about it

- The MLP's accuracy on **shifted** galaxies drops toward chance (~10% for 10
  classes). Why is it *specifically* the fully-connected first layer that makes it
  position-dependent?
- The CNN holds up because a feature detected at one location uses the **same
  weights** as at any other. Which layers give it that property — the convolutions,
  the pooling, or both?
- Try `FAST = False` (64x64). The MLP's parameter count explodes; the CNN's grows
  only a little. Which term in each count is responsible?
- Real Galaxy10 images are already centred, so we had to shift them by hand. In a
  real survey, where does natural translation come from? (Hint: source position in
  the cutout, dithering, mosaicking.)

---

## Exercise — Implement `conv2d_step`

The core computation of a convolutional layer at a single spatial position is an
element-wise multiply-and-sum between a patch of the image and the kernel.
This is also called the **inner product** or **dot product** of two matrices
(after flattening).

Implement `conv2d_step(image_patch, kernel)` for one position.
The test cell will call your function at every position of a 5x5 patch
and compare the result to `F.conv2d`.

In [ ]:
def conv2d_step(image_patch, kernel):
    """
    Compute the convolution output at a single spatial position.

    Args:
        image_patch : np.ndarray of shape (k, k) -- image values at this position
        kernel      : np.ndarray of shape (k, k) -- convolution kernel
    Returns:
        float -- the convolution response at this position
    """
    # Hint: element-wise multiply image_patch and kernel, then sum everything.
    # One line: np.sum(image_patch * kernel)
    raise NotImplementedError('Fill in conv2d_step')

In [ ]:
# ── Test: apply conv2d_step at every position; compare to F.conv2d ───────────
patch = GALAXY[28:33, 28:33].copy()   # 5x5 patch from the galaxy centre
k_test = KERNELS_2D['edge_h']         # 3x3 edge kernel

# Your implementation: slide the 3x3 kernel across the 5x5 patch -> 3x3 output
k = k_test.shape[0]
out_yours = np.zeros((3, 3))
for i in range(3):
    for j in range(3):
        out_yours[i, j] = conv2d_step(patch[i:i + k, j:j + k], k_test)

# PyTorch reference
patch_t = torch.tensor(patch, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
k_t     = torch.tensor(k_test, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
out_ref = F.conv2d(patch_t, k_t).squeeze().numpy()

print('Your output:')
print(np.round(out_yours, 4))
print('F.conv2d reference:')
print(np.round(out_ref, 4))

assert np.allclose(out_yours, out_ref, atol=1e-5), \
    'Results do not match -- check your multiply-and-sum.'
print('\u2713 Correct.')

---

## Going further

### Part A — Go deeper

Edge detection is the starting point, but filters can be oriented at any angle.
Biological visual cortex and trained CNNs both learn filters at multiple orientations —
not just horizontal and vertical.

**Challenge:** Implement a Gabor filter parameterised by orientation `theta` and
spatial frequency `lam`. A 2D Gabor is:

```
G(x, y) = exp(-(x_prime**2 + gamma**2 * y_prime**2) / (2 * sigma**2))
           * cos(2 * pi * x_prime / lam)

where:  x_prime = x*cos(theta) + y*sin(theta)
        y_prime = -x*sin(theta) + y*cos(theta)
```

Start with `sigma=2, gamma=0.5, lam=4` and apply the filter to `GALAXY` at four
orientations: 0, 45, 90, 135 degrees. Which orientation gives the strongest
response on the spiral arms? Can you estimate the dominant arm angle from the
response maps?

*Suggested approach:* Build the kernel on a small grid (e.g. 15x15), then use
`F.conv2d` (or `scipy.ndimage.convolve`) to apply it. Plot the four feature maps
side by side and compute the mean response magnitude for each.

### Part B — Lead forward

A CNN processes each position's neighbourhood independently.
Pixels far apart cannot influence each other directly —
information must pass through many convolutional layers to mix.

**Challenge:** With a 3x3 conv filter, how many layers are needed before information
from one side of a 64x64 galaxy image can influence the output for the opposite side?
(The *receptive field* grows by 2 pixels per layer.)
Is there a single operation that would give every output position direct access to
every input position — in one step?

*Suggested approach:* Compute the receptive field size after $N$ layers of 3x3 convolution:
`RF(N) = 1 + 2*N`. Solve for `N` such that `RF(N) >= 64`. Then think: what operation
computes a weighted sum over the *entire* input for each output position?
This is the concept introduced in the next slide deck.

---

### References

| | |
|---|---|
| Primary | Zeiler & Fergus (2013) — *Visualizing and Understanding Convolutional Networks*. ECCV. [arxiv.org/abs/1311.2901](https://arxiv.org/abs/1311.2901) |
| Primary | LeCun et al. (1998) — *Gradient-based learning applied to document recognition*. Proc. IEEE. [yann.lecun.com/exdb/publis/pdf/lecun-98.pdf](http://yann.lecun.com/exdb/publis/pdf/lecun-98.pdf) |
| Video | 3Blue1Brown — *But what is a convolution?* [youtube.com/watch?v=KuXjwB4LzSA](https://www.youtube.com/watch?v=KuXjwB4LzSA) |

---

## Solution — `conv2d_step`

A single convolution output at one spatial position is just an element-wise multiply followed by a sum — the **inner product** (or dot product) of the kernel and the image patch it covers. That is all a convolutional neuron computes.

In [ ]:
def conv2d_step(image_patch, kernel):
    # Element-wise multiply the patch with the kernel, then sum all values.
    # This is the fundamental operation of a convolutional layer — repeated
    # at every (i, j) position across the image to produce one feature map.
    return float(np.sum(image_patch * kernel))

# ── What this connects back to ────────────────────────────────────────────
# Part 1 widget: the orange highlighted window shows exactly the image_patch
# argument — the kernel slides to each position and this function runs once.
#
# Part 3 feature maps: five different kernels each run this function at every
# position independently, producing five feature maps in parallel — that is
# what Conv2d(in_channels=1, out_channels=5, kernel_size=3) computes.
#
# Part 6 receptive field: stacking conv2d_step calls means each output depends
# on a growing patch of the original image — the receptive field.
